In [1]:
"""
Scikit-Learn Model for COPD Prediction
"""

import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

from data_prep import prepare_data, DISEASE_SHORT_NAMES


# =============================================================================
# 1. LOAD DATA
# =============================================================================
TARGET = 'Maladie pulmonaire obstructive chronique'  # COPD

data = prepare_data(target_disease=TARGET)

print(f"Target: {DISEASE_SHORT_NAMES[TARGET]}")
print(f"Training samples: {data.n_train_samples}")
print(f"Test samples: {data.n_test_samples}")
print(f"Features: {data.feature_names}")


# =============================================================================
# 2. TRAIN MODELS
# =============================================================================
models = {
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
}

print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

results = {}
for name, model in models.items():
    # Train
    model.fit(data.X_train, data.y_train)
    
    # Predict
    y_pred = model.predict(data.X_test)
    
    # Evaluate
    r2 = r2_score(data.y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(data.y_test, y_pred))
    
    results[name] = {'model': model, 'r2': r2, 'rmse': rmse}
    print(f"{name:20s} | R² = {r2:.4f} | RMSE = {rmse:.1f}")

# Best model
best_name = max(results, key=lambda k: results[k]['r2'])
print(f"\n→ Best: {best_name} (R² = {results[best_name]['r2']:.4f})")


Target: COPD
Training samples: 5897
Test samples: 1475
Features: ['bc', 'co', 'nh3', 'nmvoc', 'nox', 'oc', 'pm10', 'pm25', 'so2', 'year']

MODEL COMPARISON
Ridge                | R² = 0.1564 | RMSE = 9188.9
Random Forest        | R² = 0.7373 | RMSE = 5128.3
Gradient Boosting    | R² = 0.7715 | RMSE = 4782.2

→ Best: Gradient Boosting (R² = 0.7715)


In [5]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

from data_prep import prepare_data, DISEASES, DISEASE_SHORT_NAMES

# =============================================================================
# LAG ANALYSIS: Find optimal lag for each disease
# =============================================================================
print("LAG ANALYSIS BY DISEASE")
print("="*60)

for rs in [1, 10, 30, 84]:
    print(f"\nRandom State: {rs}")
    for disease in DISEASES:
        short_name = DISEASE_SHORT_NAMES[disease]
        print(f"\n{short_name}:")
        
        for lag in [0, 3, 6, 9]:
            data = prepare_data(target_disease=disease, lag_years=lag, random_state=None)
            
            model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=rs)
            model.fit(data.X_train, data.y_train)
            
            r2 = r2_score(data.y_test, model.predict(data.X_test))
            print(f"  Lag {lag} years | R² = {r2:.4f}")


LAG ANALYSIS BY DISEASE

Random State: 1

Lung Cancer:
  Lag 0 years | R² = 0.8567
  Lag 3 years | R² = 0.8625
  Lag 6 years | R² = 0.8720
  Lag 9 years | R² = 0.8660

Pneumoconiosis:
  Lag 0 years | R² = 0.8058
  Lag 3 years | R² = 0.7868
  Lag 6 years | R² = 0.7367
  Lag 9 years | R² = 0.7928

COPD:
  Lag 0 years | R² = 0.7937
  Lag 3 years | R² = 0.7811
  Lag 6 years | R² = 0.7924
  Lag 9 years | R² = 0.8119

Asthma:
  Lag 0 years | R² = 0.8470
  Lag 3 years | R² = 0.8389
  Lag 6 years | R² = 0.7870
  Lag 9 years | R² = 0.8326

Other Chronic Resp.:
  Lag 0 years | R² = 0.6968
  Lag 3 years | R² = 0.6897
  Lag 6 years | R² = 0.7314
  Lag 9 years | R² = 0.7244

Random State: 10

Lung Cancer:
  Lag 0 years | R² = 0.8617
  Lag 3 years | R² = 0.8615
  Lag 6 years | R² = 0.8646
  Lag 9 years | R² = 0.8814

Pneumoconiosis:
  Lag 0 years | R² = 0.7652
  Lag 3 years | R² = 0.5429
  Lag 6 years | R² = 0.6877
  Lag 9 years | R² = 0.8087

COPD:
  Lag 0 years | R² = 0.7578
  Lag 3 years | R² = 0

The lag depends on the disease, but it depends a lot on the random state of the model. Pneumoconiosis and other diseases have better results with 6-9 years, but pneumocnoniosis also has a high r2 at 0. Asthma is usally stable, same for COPD and lung cancer. The lag is pretty unstable, so we'll scrap that idea.